# CSV Data Quality Analysis & Feature Extraction

This notebook performs comprehensive data quality analysis and creates analysis-ready features for tutor ranking.


In [14]:
import pandas as pd
import numpy as np
import re

# Load the CSV file
df = pd.read_csv('data.csv')

print("=== BASIC DATA INSPECTION ===")
print(f"Dataset shape: {df.shape}")
print(f"Columns: {list(df.columns)}")
print("\n=== DATA INFO ===")
df.info()


=== BASIC DATA INSPECTION ===
Dataset shape: (100, 8)
Columns: ['tutor_id', 'name', 'city', 'subjects', 'years_experience', 'hourly_rate', 'rating', 'bio']

=== DATA INFO ===
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100 entries, 0 to 99
Data columns (total 8 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   tutor_id          100 non-null    int64  
 1   name              100 non-null    object 
 2   city              100 non-null    object 
 3   subjects          97 non-null     object 
 4   years_experience  99 non-null     object 
 5   hourly_rate       96 non-null     object 
 6   rating            81 non-null     float64
 7   bio               100 non-null    object 
dtypes: float64(1), int64(1), object(6)
memory usage: 6.4+ KB


In [15]:
print("=== FIRST 10 ROWS ===")
df.head(10)


=== FIRST 10 ROWS ===


,tutor_id,name,city,subjects,years_experience,hourly_rate,rating,bio
0,9,Ali Yousaf,karachi,Social Studies,3,35,3.4,Passionate about student success with individu...
1,71,Zain Abbasi,RAWALPINDI,GEOGRAPHY,10,46,5.2,Passionate about student success with individu...
2,83,Daniyal Farooq,lahore,Urdu,12,NaN,4.5,Passionate about student success with individu...
3,29,Ahmad Imran,Karachi,Finance Basics,3,NaN,NaN,Passionate about student success with individu...
4,64,Laiba Yousaf,karachi,Earth Science,5+,PKR 3360,4.9,"Bilingual; comfortable with ICSE, O/A Levels, ..."
5,1,Hassan Khan,Lahore,"History , phy, Biology",seven,PKR 7840,3.9,Exam strategy specialist; past students improv...
6,6,Ahmad Hameed,Faisalabad,"Life Sciences , COMPUTER SCIENCE , ACC...",1,PKR 2800,NaN,"Bilingual; comfortable with ICSE, O/A Levels, ..."
7,51,Hassan Nawaz,Lahore,NaN,3,45usd,4.1,"Bilingual; comfortable with ICSE, O/A Levels, ..."
8,82,Laiba Javed,faisalabad,"LANGUAGE ARTS ,,,CHEMISTRY, Biology",4,$18,4.4,Exam strategy specialist; past students improv...
9,5,Anaya Khan,RAWALPINDI,Urdu,4,"34,-",NaN,Flexible scheduling; online and in-person opti...


In [16]:
print("=== YEARS_EXPERIENCE ANALYSIS ===")
print("Unique values in years_experience:")
print(df['years_experience'].value_counts(dropna=False))
print(f"\nNon-numeric values found:")
non_numeric_years = df[~df['years_experience'].astype(str).str.match(r'^\d+$', na=False)]
print(non_numeric_years[['tutor_id', 'name', 'years_experience']])


=== YEARS_EXPERIENCE ANALYSIS ===
Unique values in years_experience:
years_experience
4           12
5           11
2            9
3            8
3+           5
8            5
1            5
9            4
6            4
7            3
12           3
10           3
9+           2
2+           2
15           2
—            2
5+           2
2 years      2
NaN          1
3 years      1
12 years     1
7+           1
eight        1
0            1
6 years      1
8+           1
four         1
0+           1
six          1
1 years      1
4 years      1
three        1
seven        1
1+           1
Name: count, dtype: int64

Non-numeric values found:
    tutor_id            name years_experience
4         64    Laiba Yousaf               5+
5          1     Hassan Khan            seven
12        77     Mariam Butt          2 years
15        43     Amna Abbasi                —
23        30      Saad Nawaz               2+
27        10    Anaya Abbasi               8+
29        42   Hassan Yousaf 

In [17]:
print("=== HOURLY_RATE ANALYSIS ===")
print("Unique values in hourly_rate:")
print(df['hourly_rate'].value_counts(dropna=False))
print(f"\nValues with currency symbols:")
currency_values = df[df['hourly_rate'].astype(str).str.contains(r'[$PKR]', na=False)]
print(currency_values[['tutor_id', 'name', 'hourly_rate']])


=== HOURLY_RATE ANALYSIS ===
Unique values in hourly_rate:
hourly_rate
NaN         4
26          4
 8          3
$39         3
19          3
           ..
$21         1
 27         1
$25         1
25          1
PKR 8680    1
Name: count, Length: 75, dtype: int64

Values with currency symbols:
    tutor_id            name hourly_rate
4         64    Laiba Yousaf    PKR 3360
5          1     Hassan Khan    PKR 7840
6          6    Ahmad Hameed    PKR 2800
8         82     Laiba Javed         $18
11        66     Omar Khalid         $49
13        61    Hassan Malik         $30
16        78    Fatima Malik   PKR 11200
17        39        Ali Khan         $43
21        89     Usman Imran         $12
22        20        Ali Khan         $42
29        42   Hassan Yousaf         $38
30        22       Usman Mir         $22
31        91      Amna Anwar   PKR 11760
40        21      Omar Javed         $21
44         2      Amna Anwar   PKR 12040
45        75    Noor Ibrahim   PKR 12040
46       

In [18]:
print("=== RATING ANALYSIS ===")
print("Unique values in rating:")
print(df['rating'].value_counts(dropna=False))
print(f"\nMissing ratings:")
missing_ratings = df[df['rating'].isna()]
print(f"Count of missing ratings: {len(missing_ratings)}")
print(missing_ratings[['tutor_id', 'name', 'rating']])
print(f"\nOut-of-range ratings (>5 or <0):")
out_of_range = df[(df['rating'] > 5) | (df['rating'] < 0)]
print(f"Count of out-of-range ratings: {len(out_of_range)}")
if len(out_of_range) > 0:
    print(out_of_range[['tutor_id', 'name', 'rating']])


=== RATING ANALYSIS ===
Unique values in rating:
rating
NaN    19
3.5     7
4.5     6
4.3     6
5.1     5
3.8     5
4.0     4
3.9     4
4.1     4
4.4     4
3.0     3
5.6     3
3.4     3
4.8     3
4.9     3
4.7     2
5.2     2
4.6     2
3.2     2
5.0     2
3.7     2
4.2     2
3.3     1
5.7     1
6.0     1
3.1     1
5.4     1
3.6     1
2.8     1
Name: count, dtype: int64

Missing ratings:
Count of missing ratings: 19
    tutor_id            name  rating
3         29     Ahmad Imran     NaN
6          6    Ahmad Hameed     NaN
9          5      Anaya Khan     NaN
25        92     Hassan Butt     NaN
33        70  Mariam Hussain     NaN
37        98    Anaya Hameed     NaN
42        38  Daniyal Khalid     NaN
51        27     Laiba Nawaz     NaN
59        44  Mariam Qureshi     NaN
71        35       Hamza Mir     NaN
72        28     Zain Yousaf     NaN
76        62      Saad Nawaz     NaN
77        94    Zain Ibrahim     NaN
84        69       Zain Khan     NaN
87        79        Sara M

In [19]:
print("=== SUBJECTS ANALYSIS ===")
print("Sample subjects with inconsistent formatting:")
sample_subjects = df['subjects'].dropna().head(20)
for i, subject in enumerate(sample_subjects):
    print(f"{i+1}. '{subject}'")

print(f"\nSubjects with extra spaces or inconsistent casing:")
inconsistent_subjects = df[df['subjects'].astype(str).str.contains(r'\s{2,}|[A-Z].*[a-z]|[a-z].*[A-Z]', na=False)]
print(f"Count of inconsistent subjects: {len(inconsistent_subjects)}")
print(inconsistent_subjects[['tutor_id', 'name', 'subjects']].head(10))


=== SUBJECTS ANALYSIS ===
Sample subjects with inconsistent formatting:
1. 'Social Studies'
2. 'GEOGRAPHY'
3. 'Urdu'
4. 'Finance Basics'
5. 'Earth Science'
6. 'History  , phy, Biology'
7. '  Life Sciences  ,    COMPUTER SCIENCE  ,  ACCOUNTING'
8. 'LANGUAGE ARTS  ,,,CHEMISTRY, Biology'
9. 'Urdu'
10. 'LANGUAGE ARTS, MATH'
11. 'Coding'
12. 'computer science'
13. ' Organic Chem '
14. 'GEOGRAPHY'
15. 'PHYSICS, math'
16. 'PHYSICS , ORGANIC CHEM'
17. 'CHEM'
18. ' LANGUAGE ARTS  , Geography'
19. 'BIOLOGY, Accounting'
20. 'PHY ,   finance basics  '

Subjects with extra spaces or inconsistent casing:
Count of inconsistent subjects: 72
    tutor_id            name  \
0          9      Ali Yousaf   
2         83  Daniyal Farooq   
3         29     Ahmad Imran   
4         64    Laiba Yousaf   
5          1     Hassan Khan   
6          6    Ahmad Hameed   
8         82     Laiba Javed   
9          5      Anaya Khan   
11        66     Omar Khalid   
13        61    Hassan Malik   

              

In [20]:
print("=== DATA CLEANING DEMONSTRATION ===")

# Create a copy for cleaning demonstration
df_clean = df.copy()

print("1. Cleaning years_experience:")
print("Before cleaning - non-numeric values:")
non_numeric_before = df_clean[~df_clean['years_experience'].astype(str).str.match(r'^\d+$', na=False)]
print(non_numeric_before['years_experience'].value_counts())

# Clean years_experience
def clean_years_experience(value):
    if pd.isna(value):
        return np.nan
    value_str = str(value).strip()
    # Handle common patterns
    if value_str in ['seven', 'seven']:
        return 7
    elif value_str in ['five', 'five']:
        return 5
    elif value_str in ['three', 'three']:
        return 3
    elif value_str in ['four', 'four']:
        return 4
    elif value_str in ['six', 'six']:
        return 6
    elif value_str in ['eight', 'eight']:
        return 8
    elif value_str == '—':
        return np.nan
    elif '+' in value_str:
        # Extract number before +
        match = re.search(r'(\d+)\+', value_str)
        if match:
            return int(match.group(1))
    elif 'years' in value_str.lower():
        # Extract number before "years"
        match = re.search(r'(\d+)\s*years?', value_str.lower())
        if match:
            return int(match.group(1))
    elif 'year' in value_str.lower():
        # Extract number before "year"
        match = re.search(r'(\d+)\s*year', value_str.lower())
        if match:
            return int(match.group(1))
    else:
        # Try to convert to numeric
        try:
            return pd.to_numeric(value_str, errors='coerce')
        except:
            return np.nan

df_clean['years_experience_cleaned'] = df_clean['years_experience'].apply(clean_years_experience)
print(f"\nAfter cleaning - non-numeric values remaining:")
non_numeric_after = df_clean[df_clean['years_experience_cleaned'].isna()]
print(f"Count: {len(non_numeric_after)}")
print(non_numeric_after[['tutor_id', 'name', 'years_experience', 'years_experience_cleaned']])


=== DATA CLEANING DEMONSTRATION ===
1. Cleaning years_experience:
Before cleaning - non-numeric values:
years_experience
3+          5
5+          2
2 years     2
—           2
2+          2
9+          2
four        1
12 years    1
7+          1
eight       1
6 years     1
3 years     1
1 years     1
0+          1
six         1
seven       1
4 years     1
three       1
8+          1
1+          1
Name: count, dtype: int64

After cleaning - non-numeric values remaining:
Count: 3
    tutor_id         name years_experience  years_experience_cleaned
15        43  Amna Abbasi                —                       NaN
61        95     Ali Khan                —                       NaN
96        84  Ahmad Javed              NaN                       NaN


In [21]:
print("2. Cleaning hourly_rate:")
print("Before cleaning - values with currency symbols:")
currency_before = df_clean[df_clean['hourly_rate'].astype(str).str.contains(r'[$PKR]', na=False)]
print(currency_before['hourly_rate'].value_counts())

# Clean hourly_rate
def clean_hourly_rate(value):
    if pd.isna(value) or str(value).strip() in ['N/A', '']:
        return np.nan
    value_str = str(value).strip()
    
    # Remove currency symbols and extract numeric value
    # Handle PKR values
    if 'PKR' in value_str.upper():
        # Extract number after PKR
        match = re.search(r'PKR\s*(\d+)', value_str.upper())
        if match:
            return int(match.group(1))
    
    # Handle USD values
    if '$' in value_str:
        # Extract number before $
        match = re.search(r'(\d+)\s*\$', value_str)
        if match:
            return int(match.group(1))
        # Or extract number after $
        match = re.search(r'\$\s*(\d+)', value_str)
        if match:
            return int(match.group(1))
    
    # Handle "usd" suffix
    if 'usd' in value_str.lower():
        match = re.search(r'(\d+)\s*usd', value_str.lower())
        if match:
            return int(match.group(1))
    
    # Handle values with commas and dashes like "31,-"
    if ',' in value_str and '-' in value_str:
        match = re.search(r'(\d+),', value_str)
        if match:
            return int(match.group(1))
    
    # Try to extract just the numeric part
    match = re.search(r'(\d+)', value_str)
    if match:
        return int(match.group(1))
    
    return np.nan

df_clean['hourly_rate_cleaned'] = df_clean['hourly_rate'].apply(clean_hourly_rate)
print(f"\nAfter cleaning - non-numeric values remaining:")
non_numeric_rate_after = df_clean[df_clean['hourly_rate_cleaned'].isna()]
print(f"Count: {len(non_numeric_rate_after)}")
print(non_numeric_rate_after[['tutor_id', 'name', 'hourly_rate', 'hourly_rate_cleaned']])


2. Cleaning hourly_rate:
Before cleaning - values with currency symbols:
hourly_rate
$39          3
PKR 3360     2
PKR 12880    2
PKR 12040    2
$38          2
$42          2
$49          2
PKR 3640     1
PKR 7000     1
$41          1
PKR 12320    1
$19          1
$37          1
$46          1
$48          1
$36          1
$29          1
PKR 9240     1
PKR 10360    1
$28          1
$16          1
PKR 6440     1
PKR 7840     1
$40          1
$25          1
$21          1
PKR 11760    1
$22          1
$12          1
$43          1
PKR 11200    1
$30          1
$18          1
PKR 2800     1
PKR 8680     1
Name: count, dtype: int64

After cleaning - non-numeric values remaining:
Count: 4
    tutor_id            name hourly_rate  hourly_rate_cleaned
2         83  Daniyal Farooq         NaN                  NaN
3         29     Ahmad Imran         NaN                  NaN
18        57     Amna Sheikh         NaN                  NaN
99        67   Hassan Hameed         NaN                  N

In [22]:
print("=== FEATURE EXTRACTION ===")

# Create a clean dataset for feature extraction
df_features = df_clean.copy()

print("1. Creating skills_set feature:")
print("Sample subjects before processing:")
print(df_features['subjects'].head(5).tolist())

def create_skills_set(subjects_str):
    """Convert subjects string to a set of individual skills"""
    if pd.isna(subjects_str) or str(subjects_str).strip() == '':
        return set()
    
    # Split by comma and clean each subject
    subjects = [subject.strip() for subject in str(subjects_str).split(',')]
    
    # Clean and normalize each subject
    skills = set()
    for subject in subjects:
        if subject and subject != '':
            # Remove extra spaces, normalize casing
            cleaned_subject = ' '.join(subject.split()).title()
            skills.add(cleaned_subject)
    
    return skills

df_features['skills_set'] = df_features['subjects'].apply(create_skills_set)

print("\nSample skills_set after processing:")
for i, skills in enumerate(df_features['skills_set'].head(5)):
    print(f"{i+1}. {skills}")

print(f"\nTotal unique skills found: {len(set().union(*df_features['skills_set']))}")
print("All unique skills:")
all_skills = sorted(set().union(*df_features['skills_set']))
for skill in all_skills:
    print(f"  - {skill}")


=== FEATURE EXTRACTION ===
1. Creating skills_set feature:
Sample subjects before processing:
['Social Studies', 'GEOGRAPHY', 'Urdu', 'Finance Basics', 'Earth Science']

Sample skills_set after processing:
1. {'Social Studies'}
2. {'Geography'}
3. {'Urdu'}
4. {'Finance Basics'}
5. {'Earth Science'}

Total unique skills found: 31
All unique skills:
  - Accounting
  - Accounts
  - Algebra
  - Bio
  - Biology
  - Chem
  - Chemistry
  - Coding
  - Computer Science
  - Cs
  - Earth Science
  - Eng
  - English
  - Finance Basics
  - Geo
  - Geography
  - History
  - Language Arts
  - Life Sciences
  - Literature
  - Math
  - Mathematics
  - Maths
  - Mechanics
  - Organic Chem
  - Pak Studies
  - Phy
  - Physics
  - Programming
  - Social Studies
  - Urdu


In [23]:
print("2. Creating years_num feature:")
print("Sample years_experience before processing:")
print(df_features['years_experience'].head(10).tolist())

def create_years_num(value):
    """Convert years_experience to integer with default 0"""
    if pd.isna(value):
        return 0
    
    value_str = str(value).strip()
    
    # Handle common patterns
    if value_str in ['seven', 'seven']:
        return 7
    elif value_str in ['five', 'five']:
        return 5
    elif value_str in ['three', 'three']:
        return 3
    elif value_str in ['four', 'four']:
        return 4
    elif value_str in ['six', 'six']:
        return 6
    elif value_str in ['eight', 'eight']:
        return 8
    elif value_str == '—':
        return 0
    elif '+' in value_str:
        # Extract number before +
        match = re.search(r'(\d+)\+', value_str)
        if match:
            return int(match.group(1))
    elif 'years' in value_str.lower():
        # Extract number before "years"
        match = re.search(r'(\d+)\s*years?', value_str.lower())
        if match:
            return int(match.group(1))
    elif 'year' in value_str.lower():
        # Extract number before "year"
        match = re.search(r'(\d+)\s*year', value_str.lower())
        if match:
            return int(match.group(1))
    else:
        # Try to convert to numeric
        try:
            result = pd.to_numeric(value_str, errors='coerce')
            return int(result) if not pd.isna(result) else 0
        except:
            return 0

df_features['years_num'] = df_features['years_experience'].apply(create_years_num)

print("\nSample years_num after processing:")
print(df_features[['tutor_id', 'name', 'years_experience', 'years_num']].head(10))

print(f"\nYears_num statistics:")
print(f"Min: {df_features['years_num'].min()}")
print(f"Max: {df_features['years_num'].max()}")
print(f"Mean: {df_features['years_num'].mean():.1f}")
print(f"Median: {df_features['years_num'].median()}")
print(f"Default (0) values: {(df_features['years_num'] == 0).sum()}")


2. Creating years_num feature:
Sample years_experience before processing:
['3', '10', '12', '3', '5+', 'seven', '1', '3', '4', '4']

Sample years_num after processing:
   tutor_id            name years_experience  years_num
0         9      Ali Yousaf                3          3
1        71     Zain Abbasi               10         10
2        83  Daniyal Farooq               12         12
3        29     Ahmad Imran                3          3
4        64    Laiba Yousaf               5+          5
5         1     Hassan Khan            seven          7
6         6    Ahmad Hameed                1          1
7        51    Hassan Nawaz                3          3
8        82     Laiba Javed                4          4
9         5      Anaya Khan                4          4

Years_num statistics:
Min: 0
Max: 15
Mean: 4.9
Median: 4.0
Default (0) values: 5


In [24]:
print("3. Creating rate_num feature:")
print("Sample hourly_rate before processing:")
print(df_features['hourly_rate'].head(10).tolist())

def create_rate_num(value):
    """Convert hourly_rate to numeric with proper handling of currency symbols"""
    if pd.isna(value) or str(value).strip() in ['N/A', '']:
        return np.nan
    
    value_str = str(value).strip()
    
    # Remove currency symbols and extract numeric value
    # Handle PKR values
    if 'PKR' in value_str.upper():
        # Extract number after PKR
        match = re.search(r'PKR\s*(\d+)', value_str.upper())
        if match:
            return float(match.group(1))
    
    # Handle USD values
    if '$' in value_str:
        # Extract number before $
        match = re.search(r'(\d+)\s*\$', value_str)
        if match:
            return float(match.group(1))
        # Or extract number after $
        match = re.search(r'\$\s*(\d+)', value_str)
        if match:
            return float(match.group(1))
    
    # Handle "usd" suffix
    if 'usd' in value_str.lower():
        match = re.search(r'(\d+)\s*usd', value_str.lower())
        if match:
            return float(match.group(1))
    
    # Handle values with commas and dashes like "31,-"
    if ',' in value_str and '-' in value_str:
        match = re.search(r'(\d+),', value_str)
        if match:
            return float(match.group(1))
    
    # Try to extract just the numeric part
    match = re.search(r'(\d+)', value_str)
    if match:
        return float(match.group(1))
    
    return np.nan

df_features['rate_num'] = df_features['hourly_rate'].apply(create_rate_num)

print("\nSample rate_num after processing:")
print(df_features[['tutor_id', 'name', 'hourly_rate', 'rate_num']].head(10))

print(f"\nRate_num statistics:")
print(f"Min: {df_features['rate_num'].min()}")
print(f"Max: {df_features['rate_num'].max()}")
print(f"Mean: {df_features['rate_num'].mean():.1f}")
print(f"Median: {df_features['rate_num'].median()}")
print(f"Missing values: {df_features['rate_num'].isna().sum()}")
print(f"Non-missing values: {df_features['rate_num'].notna().sum()}")


3. Creating rate_num feature:
Sample hourly_rate before processing:
[' 35 ', '46', nan, nan, 'PKR 3360', 'PKR 7840', 'PKR 2800', '45usd', '$18', '34,-']

Sample rate_num after processing:
   tutor_id            name hourly_rate  rate_num
0         9      Ali Yousaf         35       35.0
1        71     Zain Abbasi          46      46.0
2        83  Daniyal Farooq         NaN       NaN
3        29     Ahmad Imran         NaN       NaN
4        64    Laiba Yousaf    PKR 3360    3360.0
5         1     Hassan Khan    PKR 7840    7840.0
6         6    Ahmad Hameed    PKR 2800    2800.0
7        51    Hassan Nawaz       45usd      45.0
8        82     Laiba Javed         $18      18.0
9         5      Anaya Khan        34,-      34.0

Rate_num statistics:
Min: 8.0
Max: 12880.0
Mean: 1564.2
Median: 33.5
Missing values: 4
Non-missing values: 96


In [25]:
print("4. Creating rating_norm feature (scaled to [0,1]):")
print("Sample rating before processing:")
print(df_features['rating'].head(10).tolist())

def create_rating_norm(value):
    """Scale rating to [0,1] range"""
    if pd.isna(value):
        return np.nan
    
    # Assuming rating scale is 0-5, scale to 0-1
    # If rating is > 5, cap it at 5
    normalized = min(value, 5.0) / 5.0
    return normalized

df_features['rating_norm'] = df_features['rating'].apply(create_rating_norm)

print("\nSample rating_norm after processing:")
print(df_features[['tutor_id', 'name', 'rating', 'rating_norm']].head(10))

print(f"\nRating_norm statistics:")
print(f"Min: {df_features['rating_norm'].min()}")
print(f"Max: {df_features['rating_norm'].max()}")
print(f"Mean: {df_features['rating_norm'].mean():.3f}")
print(f"Median: {df_features['rating_norm'].median():.3f}")
print(f"Missing values: {df_features['rating_norm'].isna().sum()}")

print(f"\nOriginal rating distribution:")
print(df_features['rating'].value_counts().sort_index())


4. Creating rating_norm feature (scaled to [0,1]):
Sample rating before processing:
[3.4, 5.2, 4.5, nan, 4.9, 3.9, nan, 4.1, 4.4, nan]

Sample rating_norm after processing:
   tutor_id            name  rating  rating_norm
0         9      Ali Yousaf     3.4         0.68
1        71     Zain Abbasi     5.2         1.00
2        83  Daniyal Farooq     4.5         0.90
3        29     Ahmad Imran     NaN          NaN
4        64    Laiba Yousaf     4.9         0.98
5         1     Hassan Khan     3.9         0.78
6         6    Ahmad Hameed     NaN          NaN
7        51    Hassan Nawaz     4.1         0.82
8        82     Laiba Javed     4.4         0.88
9         5      Anaya Khan     NaN          NaN

Rating_norm statistics:
Min: 0.5599999999999999
Max: 1.0
Mean: 0.837
Median: 0.860
Missing values: 19

Original rating distribution:
rating
2.8    1
3.0    3
3.1    1
3.2    2
3.3    1
3.4    3
3.5    7
3.6    1
3.7    2
3.8    5
3.9    4
4.0    4
4.1    4
4.2    2
4.3    6
4.4    4
4.5

In [26]:
print("=== FEATURE ANALYSIS FOR RANKING ===")

# Create additional derived features for ranking analysis
df_features['num_skills'] = df_features['skills_set'].apply(len)
df_features['has_rating'] = df_features['rating'].notna()
df_features['has_rate'] = df_features['rate_num'].notna()

print("1. Feature Correlation Analysis:")
# Select numeric features for correlation
numeric_features = ['years_num', 'rate_num', 'rating_norm', 'num_skills']
correlation_data = df_features[numeric_features].corr()
print(correlation_data)

print("\n2. Feature Distribution Analysis:")
print("Years of Experience:")
print(f"  - 0-2 years: {(df_features['years_num'] <= 2).sum()} tutors")
print(f"  - 3-5 years: {((df_features['years_num'] >= 3) & (df_features['years_num'] <= 5)).sum()} tutors")
print(f"  - 6+ years: {(df_features['years_num'] >= 6).sum()} tutors")

print("\nHourly Rate:")
print(f"  - Low rate (≤20): {(df_features['rate_num'] <= 20).sum()} tutors")
print(f"  - Medium rate (21-40): {((df_features['rate_num'] >= 21) & (df_features['rate_num'] <= 40)).sum()} tutors")
print(f"  - High rate (≥41): {(df_features['rate_num'] >= 41).sum()} tutors")

print("\nRating:")
print(f"  - High rating (≥4.0): {(df_features['rating_norm'] >= 0.8).sum()} tutors")
print(f"  - Medium rating (3.0-3.9): {((df_features['rating_norm'] >= 0.6) & (df_features['rating_norm'] < 0.8)).sum()} tutors")
print(f"  - Low rating (<3.0): {(df_features['rating_norm'] < 0.6).sum()} tutors")

print("\nNumber of Skills:")
print(f"  - Single skill: {(df_features['num_skills'] == 1).sum()} tutors")
print(f"  - 2-3 skills: {((df_features['num_skills'] >= 2) & (df_features['num_skills'] <= 3)).sum()} tutors")
print(f"  - 4+ skills: {(df_features['num_skills'] >= 4).sum()} tutors")


=== FEATURE ANALYSIS FOR RANKING ===
1. Feature Correlation Analysis:
             years_num  rate_num  rating_norm  num_skills
years_num     1.000000 -0.008909    -0.117171   -0.122824
rate_num     -0.008909  1.000000    -0.015839    0.161822
rating_norm  -0.117171 -0.015839     1.000000   -0.101152
num_skills   -0.122824  0.161822    -0.101152    1.000000

2. Feature Distribution Analysis:
Years of Experience:
  - 0-2 years: 25 tutors
  - 3-5 years: 42 tutors
  - 6+ years: 33 tutors

Hourly Rate:
  - Low rate (≤20): 22 tutors
  - Medium rate (21-40): 39 tutors
  - High rate (≥41): 35 tutors

Rating:
  - High rating (≥4.0): 51 tutors
  - Medium rating (3.0-3.9): 29 tutors
  - Low rating (<3.0): 1 tutors

Number of Skills:
  - Single skill: 38 tutors
  - 2-3 skills: 59 tutors
  - 4+ skills: 0 tutors


In [27]:
print("=== FEATURE IMPORTANCE FOR RANKING ===")

print("1. RATING_NORM - Most Important for Quality Assessment:")
print("   - Direct measure of tutor performance")
print("   - Scaled to [0,1] for easy comparison")
print("   - Missing values need imputation strategy")
print(f"   - Coverage: {df_features['rating_norm'].notna().sum()}/{len(df_features)} ({df_features['rating_norm'].notna().sum()/len(df_features)*100:.1f}%)")

print("\n2. YEARS_NUM - Important for Experience Assessment:")
print("   - Indicates teaching experience and expertise")
print("   - Integer values with default 0 for missing")
print("   - Strong predictor of teaching quality")
print(f"   - Coverage: {df_features['years_num'].notna().sum()}/{len(df_features)} ({df_features['years_num'].notna().sum()/len(df_features)*100:.1f}%)")

print("\n3. RATE_NUM - Important for Pricing Analysis:")
print("   - Market rate indicator")
print("   - Helps assess value proposition")
print("   - Missing values indicate pricing not disclosed")
print(f"   - Coverage: {df_features['rate_num'].notna().sum()}/{len(df_features)} ({df_features['rate_num'].notna().sum()/len(df_features)*100:.1f}%)")

print("\n4. SKILLS_SET - Important for Subject Matching:")
print("   - Enables subject-based filtering")
print("   - Shows teaching versatility")
print("   - Critical for tutor-student matching")
print(f"   - Coverage: {df_features['num_skills'].gt(0).sum()}/{len(df_features)} ({df_features['num_skills'].gt(0).sum()/len(df_features)*100:.1f}%)")

print("\n=== RANKING STRATEGY RECOMMENDATIONS ===")
print("Primary Ranking Factors (in order of importance):")
print("1. RATING_NORM - Quality indicator")
print("2. YEARS_NUM - Experience indicator") 
print("3. NUM_SKILLS - Versatility indicator")
print("4. RATE_NUM - Value indicator (lower is better for students)")

print("\nSecondary Factors:")
print("- City location for geographic matching")
print("- Subject specialization for skill matching")
print("- Availability indicators")


=== FEATURE IMPORTANCE FOR RANKING ===
1. RATING_NORM - Most Important for Quality Assessment:
   - Direct measure of tutor performance
   - Scaled to [0,1] for easy comparison
   - Missing values need imputation strategy
   - Coverage: 81/100 (81.0%)

2. YEARS_NUM - Important for Experience Assessment:
   - Indicates teaching experience and expertise
   - Integer values with default 0 for missing
   - Strong predictor of teaching quality
   - Coverage: 100/100 (100.0%)

3. RATE_NUM - Important for Pricing Analysis:
   - Market rate indicator
   - Helps assess value proposition
   - Missing values indicate pricing not disclosed
   - Coverage: 96/100 (96.0%)

4. SKILLS_SET - Important for Subject Matching:
   - Enables subject-based filtering
   - Shows teaching versatility
   - Critical for tutor-student matching
   - Coverage: 97/100 (97.0%)

=== RANKING STRATEGY RECOMMENDATIONS ===
Primary Ranking Factors (in order of importance):
1. RATING_NORM - Quality indicator
2. YEARS_NUM - Exp

In [28]:
print("=== COLUMN STRATEGY RECOMMENDATION ===")

print("RECOMMENDATION: Keep BOTH raw and cleaned columns")
print("\nRationale:")

print("\n1. RAW COLUMNS (Keep for):")
print("   - Audit trail and data provenance")
print("   - Debugging data quality issues")
print("   - Understanding original data patterns")
print("   - Compliance and transparency")
print("   - Future re-processing with different logic")

print("\n2. CLEANED COLUMNS (Use for):")
print("   - Analysis and modeling")
print("   - Ranking algorithms")
print("   - Visualization")
print("   - Machine learning features")
print("   - Business logic implementation")

print("\n3. STORAGE STRATEGY:")
print("   - Keep raw columns in 'raw_' prefix")
print("   - Use cleaned columns as primary features")
print("   - Document transformation logic")
print("   - Version control data cleaning functions")

print("\n4. EXAMPLE COLUMN STRUCTURE:")
print("   Raw columns:     Cleaned columns:")
print("   - subjects    →  - skills_set")
print("   - years_experience → - years_num") 
print("   - hourly_rate →  - rate_num")
print("   - rating      →  - rating_norm")

print("\n5. BENEFITS:")
print("   - Data lineage tracking")
print("   - Easy rollback if needed")
print("   - Transparent data processing")
print("   - Regulatory compliance")
print("   - Stakeholder confidence")

print("\n6. IMPLEMENTATION:")
print("   - Store both in same DataFrame")
print("   - Use naming convention (raw_* vs clean_*)")
print("   - Document transformation functions")
print("   - Create data dictionary")


=== COLUMN STRATEGY RECOMMENDATION ===
RECOMMENDATION: Keep BOTH raw and cleaned columns

Rationale:

1. RAW COLUMNS (Keep for):
   - Audit trail and data provenance
   - Debugging data quality issues
   - Understanding original data patterns
   - Compliance and transparency
   - Future re-processing with different logic

2. CLEANED COLUMNS (Use for):
   - Analysis and modeling
   - Ranking algorithms
   - Visualization
   - Machine learning features
   - Business logic implementation

3. STORAGE STRATEGY:
   - Keep raw columns in 'raw_' prefix
   - Use cleaned columns as primary features
   - Document transformation logic
   - Version control data cleaning functions

4. EXAMPLE COLUMN STRUCTURE:
   Raw columns:     Cleaned columns:
   - subjects    →  - skills_set
   - years_experience → - years_num
   - hourly_rate →  - rate_num
   - rating      →  - rating_norm

5. BENEFITS:
   - Data lineage tracking
   - Easy rollback if needed
   - Transparent data processing
   - Regulatory comp

In [29]:
print("=== FINAL FEATURE EXTRACTION SUMMARY ===")

print("Successfully created analysis-ready features:")
print("\n1. SKILLS_SET:")
print(f"   - Type: Set of strings")
print(f"   - Unique skills: {len(set().union(*df_features['skills_set']))}")
print(f"   - Coverage: {df_features['num_skills'].gt(0).sum()}/{len(df_features)} tutors")

print("\n2. YEARS_NUM:")
print(f"   - Type: Integer")
print(f"   - Range: {df_features['years_num'].min()} - {df_features['years_num'].max()}")
print(f"   - Default 0 values: {(df_features['years_num'] == 0).sum()}")

print("\n3. RATE_NUM:")
print(f"   - Type: Float")
print(f"   - Range: {df_features['rate_num'].min()} - {df_features['rate_num'].max()}")
print(f"   - Missing values: {df_features['rate_num'].isna().sum()}")

print("\n4. RATING_NORM:")
print(f"   - Type: Float (0-1 scale)")
print(f"   - Range: {df_features['rating_norm'].min():.3f} - {df_features['rating_norm'].max():.3f}")
print(f"   - Missing values: {df_features['rating_norm'].isna().sum()}")

print("\n5. DERIVED FEATURES:")
print(f"   - NUM_SKILLS: {df_features['num_skills'].min()} - {df_features['num_skills'].max()}")
print(f"   - HAS_RATING: {df_features['has_rating'].sum()} tutors")
print(f"   - HAS_RATE: {df_features['has_rate'].sum()} tutors")

print("\n=== READY FOR RANKING ALGORITHM ===")
print("Primary features for ranking:")
print("- rating_norm (quality)")
print("- years_num (experience)")
print("- num_skills (versatility)")
print("- rate_num (value - lower is better)")

print("\nNext steps:")
print("1. Handle missing values (imputation)")
print("2. Create composite ranking score")
print("3. Implement subject-based filtering")
print("4. Add geographic constraints")
print("5. Test ranking algorithm")


=== FINAL FEATURE EXTRACTION SUMMARY ===
Successfully created analysis-ready features:

1. SKILLS_SET:
   - Type: Set of strings
   - Unique skills: 31
   - Coverage: 97/100 tutors

2. YEARS_NUM:
   - Type: Integer
   - Range: 0 - 15
   - Default 0 values: 5

3. RATE_NUM:
   - Type: Float
   - Range: 8.0 - 12880.0
   - Missing values: 4

4. RATING_NORM:
   - Type: Float (0-1 scale)
   - Range: 0.560 - 1.000
   - Missing values: 19

5. DERIVED FEATURES:
   - NUM_SKILLS: 0 - 3
   - HAS_RATING: 81 tutors
   - HAS_RATE: 96 tutors

=== READY FOR RANKING ALGORITHM ===
Primary features for ranking:
- rating_norm (quality)
- years_num (experience)
- num_skills (versatility)
- rate_num (value - lower is better)

Next steps:
1. Handle missing values (imputation)
2. Create composite ranking score
3. Implement subject-based filtering
4. Add geographic constraints
5. Test ranking algorithm


In [30]:
import pandas as pd
import numpy as np
import re

# Load the CSV file
df = pd.read_csv('data.csv')

print("=== BASIC DATA INSPECTION ===")
print(f"Dataset shape: {df.shape}")
print(f"Columns: {list(df.columns)}")
print("\n=== DATA INFO ===")
df.info()


=== BASIC DATA INSPECTION ===
Dataset shape: (100, 8)
Columns: ['tutor_id', 'name', 'city', 'subjects', 'years_experience', 'hourly_rate', 'rating', 'bio']

=== DATA INFO ===
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100 entries, 0 to 99
Data columns (total 8 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   tutor_id          100 non-null    int64  
 1   name              100 non-null    object 
 2   city              100 non-null    object 
 3   subjects          97 non-null     object 
 4   years_experience  99 non-null     object 
 5   hourly_rate       96 non-null     object 
 6   rating            81 non-null     float64
 7   bio               100 non-null    object 
dtypes: float64(1), int64(1), object(6)
memory usage: 6.4+ KB


In [4]:
print("=== FIRST 10 ROWS ===")
df.head(10)


=== FIRST 10 ROWS ===


,tutor_id,name,city,subjects,years_experience,hourly_rate,rating,bio
0,9,Ali Yousaf,karachi,Social Studies,3,35,3.4,Passionate about student success with individu...
1,71,Zain Abbasi,RAWALPINDI,GEOGRAPHY,10,46,5.2,Passionate about student success with individu...
2,83,Daniyal Farooq,lahore,Urdu,12,NaN,4.5,Passionate about student success with individu...
3,29,Ahmad Imran,Karachi,Finance Basics,3,NaN,NaN,Passionate about student success with individu...
4,64,Laiba Yousaf,karachi,Earth Science,5+,PKR 3360,4.9,"Bilingual; comfortable with ICSE, O/A Levels, ..."
5,1,Hassan Khan,Lahore,"History , phy, Biology",seven,PKR 7840,3.9,Exam strategy specialist; past students improv...
6,6,Ahmad Hameed,Faisalabad,"Life Sciences , COMPUTER SCIENCE , ACC...",1,PKR 2800,NaN,"Bilingual; comfortable with ICSE, O/A Levels, ..."
7,51,Hassan Nawaz,Lahore,NaN,3,45usd,4.1,"Bilingual; comfortable with ICSE, O/A Levels, ..."
8,82,Laiba Javed,faisalabad,"LANGUAGE ARTS ,,,CHEMISTRY, Biology",4,$18,4.4,Exam strategy specialist; past students improv...
9,5,Anaya Khan,RAWALPINDI,Urdu,4,"34,-",NaN,Flexible scheduling; online and in-person opti...


In [5]:
print("=== YEARS_EXPERIENCE ANALYSIS ===")
print("Unique values in years_experience:")
print(df['years_experience'].value_counts(dropna=False))
print(f"\nNon-numeric values found:")
non_numeric_years = df[~df['years_experience'].astype(str).str.match(r'^\d+$', na=False)]
print(non_numeric_years[['tutor_id', 'name', 'years_experience']])


=== YEARS_EXPERIENCE ANALYSIS ===
Unique values in years_experience:
years_experience
4           12
5           11
2            9
3            8
3+           5
8            5
1            5
9            4
6            4
7            3
12           3
10           3
9+           2
2+           2
15           2
—            2
5+           2
2 years      2
NaN          1
3 years      1
12 years     1
7+           1
eight        1
0            1
6 years      1
8+           1
four         1
0+           1
six          1
1 years      1
4 years      1
three        1
seven        1
1+           1
Name: count, dtype: int64

Non-numeric values found:
    tutor_id            name years_experience
4         64    Laiba Yousaf               5+
5          1     Hassan Khan            seven
12        77     Mariam Butt          2 years
15        43     Amna Abbasi                —
23        30      Saad Nawaz               2+
27        10    Anaya Abbasi               8+
29        42   Hassan Yousaf 

In [6]:
print("=== HOURLY_RATE ANALYSIS ===")
print("Unique values in hourly_rate:")
print(df['hourly_rate'].value_counts(dropna=False))
print(f"\nValues with currency symbols:")
currency_values = df[df['hourly_rate'].astype(str).str.contains(r'[$PKR]', na=False)]
print(currency_values[['tutor_id', 'name', 'hourly_rate']])


=== HOURLY_RATE ANALYSIS ===
Unique values in hourly_rate:
hourly_rate
NaN         4
26          4
 8          3
$39         3
19          3
           ..
$21         1
 27         1
$25         1
25          1
PKR 8680    1
Name: count, Length: 75, dtype: int64

Values with currency symbols:
    tutor_id            name hourly_rate
4         64    Laiba Yousaf    PKR 3360
5          1     Hassan Khan    PKR 7840
6          6    Ahmad Hameed    PKR 2800
8         82     Laiba Javed         $18
11        66     Omar Khalid         $49
13        61    Hassan Malik         $30
16        78    Fatima Malik   PKR 11200
17        39        Ali Khan         $43
21        89     Usman Imran         $12
22        20        Ali Khan         $42
29        42   Hassan Yousaf         $38
30        22       Usman Mir         $22
31        91      Amna Anwar   PKR 11760
40        21      Omar Javed         $21
44         2      Amna Anwar   PKR 12040
45        75    Noor Ibrahim   PKR 12040
46       

In [7]:
print("=== RATING ANALYSIS ===")
print("Unique values in rating:")
print(df['rating'].value_counts(dropna=False))
print(f"\nMissing ratings:")
missing_ratings = df[df['rating'].isna()]
print(f"Count of missing ratings: {len(missing_ratings)}")
print(missing_ratings[['tutor_id', 'name', 'rating']])
print(f"\nOut-of-range ratings (>5 or <0):")
out_of_range = df[(df['rating'] > 5) | (df['rating'] < 0)]
print(f"Count of out-of-range ratings: {len(out_of_range)}")
if len(out_of_range) > 0:
    print(out_of_range[['tutor_id', 'name', 'rating']])


=== RATING ANALYSIS ===
Unique values in rating:
rating
NaN    19
3.5     7
4.5     6
4.3     6
5.1     5
3.8     5
4.0     4
3.9     4
4.1     4
4.4     4
3.0     3
5.6     3
3.4     3
4.8     3
4.9     3
4.7     2
5.2     2
4.6     2
3.2     2
5.0     2
3.7     2
4.2     2
3.3     1
5.7     1
6.0     1
3.1     1
5.4     1
3.6     1
2.8     1
Name: count, dtype: int64

Missing ratings:
Count of missing ratings: 19
    tutor_id            name  rating
3         29     Ahmad Imran     NaN
6          6    Ahmad Hameed     NaN
9          5      Anaya Khan     NaN
25        92     Hassan Butt     NaN
33        70  Mariam Hussain     NaN
37        98    Anaya Hameed     NaN
42        38  Daniyal Khalid     NaN
51        27     Laiba Nawaz     NaN
59        44  Mariam Qureshi     NaN
71        35       Hamza Mir     NaN
72        28     Zain Yousaf     NaN
76        62      Saad Nawaz     NaN
77        94    Zain Ibrahim     NaN
84        69       Zain Khan     NaN
87        79        Sara M

In [8]:
print("=== SUBJECTS ANALYSIS ===")
print("Sample subjects with inconsistent formatting:")
sample_subjects = df['subjects'].dropna().head(20)
for i, subject in enumerate(sample_subjects):
    print(f"{i+1}. '{subject}'")

print(f"\nSubjects with extra spaces or inconsistent casing:")
inconsistent_subjects = df[df['subjects'].astype(str).str.contains(r'\s{2,}|[A-Z].*[a-z]|[a-z].*[A-Z]', na=False)]
print(f"Count of inconsistent subjects: {len(inconsistent_subjects)}")
print(inconsistent_subjects[['tutor_id', 'name', 'subjects']].head(10))


=== SUBJECTS ANALYSIS ===
Sample subjects with inconsistent formatting:
1. 'Social Studies'
2. 'GEOGRAPHY'
3. 'Urdu'
4. 'Finance Basics'
5. 'Earth Science'
6. 'History  , phy, Biology'
7. '  Life Sciences  ,    COMPUTER SCIENCE  ,  ACCOUNTING'
8. 'LANGUAGE ARTS  ,,,CHEMISTRY, Biology'
9. 'Urdu'
10. 'LANGUAGE ARTS, MATH'
11. 'Coding'
12. 'computer science'
13. ' Organic Chem '
14. 'GEOGRAPHY'
15. 'PHYSICS, math'
16. 'PHYSICS , ORGANIC CHEM'
17. 'CHEM'
18. ' LANGUAGE ARTS  , Geography'
19. 'BIOLOGY, Accounting'
20. 'PHY ,   finance basics  '

Subjects with extra spaces or inconsistent casing:
Count of inconsistent subjects: 72
    tutor_id            name  \
0          9      Ali Yousaf   
2         83  Daniyal Farooq   
3         29     Ahmad Imran   
4         64    Laiba Yousaf   
5          1     Hassan Khan   
6          6    Ahmad Hameed   
8         82     Laiba Javed   
9          5      Anaya Khan   
11        66     Omar Khalid   
13        61    Hassan Malik   

              

In [9]:
print("=== DATA CLEANING DEMONSTRATION ===")

# Create a copy for cleaning demonstration
df_clean = df.copy()

print("1. Cleaning years_experience:")
print("Before cleaning - non-numeric values:")
non_numeric_before = df_clean[~df_clean['years_experience'].astype(str).str.match(r'^\d+$', na=False)]
print(non_numeric_before['years_experience'].value_counts())

# Clean years_experience
def clean_years_experience(value):
    if pd.isna(value):
        return np.nan
    value_str = str(value).strip()
    # Handle common patterns
    if value_str in ['seven', 'seven']:
        return 7
    elif value_str in ['five', 'five']:
        return 5
    elif value_str in ['three', 'three']:
        return 3
    elif value_str in ['four', 'four']:
        return 4
    elif value_str in ['six', 'six']:
        return 6
    elif value_str in ['eight', 'eight']:
        return 8
    elif value_str == '—':
        return np.nan
    elif '+' in value_str:
        # Extract number before +
        match = re.search(r'(\d+)\+', value_str)
        if match:
            return int(match.group(1))
    elif 'years' in value_str.lower():
        # Extract number before "years"
        match = re.search(r'(\d+)\s*years?', value_str.lower())
        if match:
            return int(match.group(1))
    elif 'year' in value_str.lower():
        # Extract number before "year"
        match = re.search(r'(\d+)\s*year', value_str.lower())
        if match:
            return int(match.group(1))
    else:
        # Try to convert to numeric
        try:
            return pd.to_numeric(value_str, errors='coerce')
        except:
            return np.nan

df_clean['years_experience_cleaned'] = df_clean['years_experience'].apply(clean_years_experience)
print(f"\nAfter cleaning - non-numeric values remaining:")
non_numeric_after = df_clean[df_clean['years_experience_cleaned'].isna()]
print(f"Count: {len(non_numeric_after)}")
print(non_numeric_after[['tutor_id', 'name', 'years_experience', 'years_experience_cleaned']])


=== DATA CLEANING DEMONSTRATION ===
1. Cleaning years_experience:
Before cleaning - non-numeric values:
years_experience
3+          5
5+          2
2 years     2
—           2
2+          2
9+          2
four        1
12 years    1
7+          1
eight       1
6 years     1
3 years     1
1 years     1
0+          1
six         1
seven       1
4 years     1
three       1
8+          1
1+          1
Name: count, dtype: int64

After cleaning - non-numeric values remaining:
Count: 3
    tutor_id         name years_experience  years_experience_cleaned
15        43  Amna Abbasi                —                       NaN
61        95     Ali Khan                —                       NaN
96        84  Ahmad Javed              NaN                       NaN


In [10]:
print("2. Cleaning hourly_rate:")
print("Before cleaning - values with currency symbols:")
currency_before = df_clean[df_clean['hourly_rate'].astype(str).str.contains(r'[$PKR]', na=False)]
print(currency_before['hourly_rate'].value_counts())

# Clean hourly_rate
def clean_hourly_rate(value):
    if pd.isna(value) or str(value).strip() in ['N/A', '']:
        return np.nan
    value_str = str(value).strip()
    
    # Remove currency symbols and extract numeric value
    # Handle PKR values
    if 'PKR' in value_str.upper():
        # Extract number after PKR
        match = re.search(r'PKR\s*(\d+)', value_str.upper())
        if match:
            return int(match.group(1))
    
    # Handle USD values
    if '$' in value_str:
        # Extract number before $
        match = re.search(r'(\d+)\s*\$', value_str)
        if match:
            return int(match.group(1))
        # Or extract number after $
        match = re.search(r'\$\s*(\d+)', value_str)
        if match:
            return int(match.group(1))
    
    # Handle "usd" suffix
    if 'usd' in value_str.lower():
        match = re.search(r'(\d+)\s*usd', value_str.lower())
        if match:
            return int(match.group(1))
    
    # Handle values with commas and dashes like "31,-"
    if ',' in value_str and '-' in value_str:
        match = re.search(r'(\d+),', value_str)
        if match:
            return int(match.group(1))
    
    # Try to extract just the numeric part
    match = re.search(r'(\d+)', value_str)
    if match:
        return int(match.group(1))
    
    return np.nan

df_clean['hourly_rate_cleaned'] = df_clean['hourly_rate'].apply(clean_hourly_rate)
print(f"\nAfter cleaning - non-numeric values remaining:")
non_numeric_rate_after = df_clean[df_clean['hourly_rate_cleaned'].isna()]
print(f"Count: {len(non_numeric_rate_after)}")
print(non_numeric_rate_after[['tutor_id', 'name', 'hourly_rate', 'hourly_rate_cleaned']])


2. Cleaning hourly_rate:
Before cleaning - values with currency symbols:
hourly_rate
$39          3
PKR 3360     2
PKR 12880    2
PKR 12040    2
$38          2
$42          2
$49          2
PKR 3640     1
PKR 7000     1
$41          1
PKR 12320    1
$19          1
$37          1
$46          1
$48          1
$36          1
$29          1
PKR 9240     1
PKR 10360    1
$28          1
$16          1
PKR 6440     1
PKR 7840     1
$40          1
$25          1
$21          1
PKR 11760    1
$22          1
$12          1
$43          1
PKR 11200    1
$30          1
$18          1
PKR 2800     1
PKR 8680     1
Name: count, dtype: int64

After cleaning - non-numeric values remaining:
Count: 4
    tutor_id            name hourly_rate  hourly_rate_cleaned
2         83  Daniyal Farooq         NaN                  NaN
3         29     Ahmad Imran         NaN                  NaN
18        57     Amna Sheikh         NaN                  NaN
99        67   Hassan Hameed         NaN                  N

In [11]:
print("3. Cleaning subjects:")
print("Before cleaning - sample subjects:")
print(df_clean['subjects'].head(10).tolist())

# Clean subjects
def clean_subjects(value):
    if pd.isna(value) or str(value).strip() == '':
        return np.nan
    
    value_str = str(value).strip()
    
    # Split by comma and clean each subject
    subjects = [subject.strip() for subject in value_str.split(',')]
    
    # Clean each subject
    cleaned_subjects = []
    for subject in subjects:
        if subject and subject != '':
            # Remove extra spaces, normalize casing
            cleaned_subject = ' '.join(subject.split()).title()
            cleaned_subjects.append(cleaned_subject)
    
    # Join back with comma
    if cleaned_subjects:
        return ', '.join(cleaned_subjects)
    else:
        return np.nan

df_clean['subjects_cleaned'] = df_clean['subjects'].apply(clean_subjects)
print(f"\nAfter cleaning - sample subjects:")
print(df_clean['subjects_cleaned'].head(10).tolist())


3. Cleaning subjects:
Before cleaning - sample subjects:
['Social Studies', 'GEOGRAPHY', 'Urdu', 'Finance Basics', 'Earth Science', 'History  , phy, Biology', '  Life Sciences  ,    COMPUTER SCIENCE  ,  ACCOUNTING', nan, 'LANGUAGE ARTS  ,,,CHEMISTRY, Biology', 'Urdu']

After cleaning - sample subjects:
['Social Studies', 'Geography', 'Urdu', 'Finance Basics', 'Earth Science', 'History, Phy, Biology', 'Life Sciences, Computer Science, Accounting', nan, 'Language Arts, Chemistry, Biology', 'Urdu']


In [12]:
print("=== PARSING FAILURE ANALYSIS ===")

# Calculate parsing failures
total_rows = len(df)
print(f"Total rows: {total_rows}")

# Years experience parsing failures
years_failures = len(df_clean[df_clean['years_experience_cleaned'].isna()])
years_failure_rate = (years_failures / total_rows) * 100
print(f"Years experience parsing failures: {years_failures} ({years_failure_rate:.1f}%)")

# Hourly rate parsing failures  
rate_failures = len(df_clean[df_clean['hourly_rate_cleaned'].isna()])
rate_failure_rate = (rate_failures / total_rows) * 100
print(f"Hourly rate parsing failures: {rate_failures} ({rate_failure_rate:.1f}%)")

# Rating issues
rating_missing = len(df[df['rating'].isna()])
rating_out_of_range = len(df[(df['rating'] > 5) | (df['rating'] < 0)])
rating_issues = rating_missing + rating_out_of_range
rating_issue_rate = (rating_issues / total_rows) * 100
print(f"Rating issues (missing + out-of-range): {rating_issues} ({rating_issue_rate:.1f}%)")

# Subjects issues
subjects_missing = len(df[df['subjects'].isna()])
subjects_issue_rate = (subjects_missing / total_rows) * 100
print(f"Missing subjects: {subjects_missing} ({subjects_issue_rate:.1f}%)")

print(f"\n=== OVERALL DATA QUALITY SUMMARY ===")
print(f"Rows with at least one parsing issue: {len(df_clean[(df_clean['years_experience_cleaned'].isna()) | (df_clean['hourly_rate_cleaned'].isna()) | (df['rating'].isna()) | (df['subjects'].isna())])}")
print(f"Percentage of rows with issues: {(len(df_clean[(df_clean['years_experience_cleaned'].isna()) | (df_clean['hourly_rate_cleaned'].isna()) | (df['rating'].isna()) | (df['subjects'].isna())]) / total_rows) * 100:.1f}%")


=== PARSING FAILURE ANALYSIS ===
Total rows: 100
Years experience parsing failures: 3 (3.0%)
Hourly rate parsing failures: 4 (4.0%)
Rating issues (missing + out-of-range): 32 (32.0%)
Missing subjects: 3 (3.0%)

=== OVERALL DATA QUALITY SUMMARY ===
Rows with at least one parsing issue: 28
Percentage of rows with issues: 28.0%


In [13]:
print("=== RECOMMENDATIONS ===")
print("\n1. YEARS_EXPERIENCE:")
print("   - Found non-numeric values like 'seven', 'five', '3+', '2 years', '—'")
print("   - Recommendation: COERCE (convert to numeric)")
print("   - Rationale: Most values can be cleaned and converted to meaningful numbers")
print("   - Use custom cleaning function to handle text numbers and extract numeric values")

print("\n2. HOURLY_RATE:")
print("   - Found currency symbols ($, PKR), suffixes (usd), and malformed values")
print("   - Recommendation: COERCE (extract numeric values)")
print("   - Rationale: Currency symbols can be stripped to get numeric values")
print("   - Use regex to extract numbers and handle different currency formats")

print("\n3. RATING:")
print("   - Found missing values and some out-of-range values (>5)")
print("   - Recommendation: COERCE for missing values, REVIEW out-of-range")
print("   - Rationale: Missing ratings could be imputed with median/mean")
print("   - Out-of-range values need manual review")

print("\n4. SUBJECTS:")
print("   - Found inconsistent casing, extra spaces, empty values")
print("   - Recommendation: COERCE (normalize formatting)")
print("   - Rationale: Text normalization preserves data while improving consistency")
print("   - Use string cleaning to standardize format")

print("\n=== STRATEGY RECOMMENDATION ===")
print("COERCE rather than DROP because:")
print("- Most parsing issues are fixable with data cleaning")
print("- Dropping rows would lose valuable data")
print("- The dataset is relatively small (100 rows)")
print("- Missing values can be handled with imputation strategies")


=== RECOMMENDATIONS ===

1. YEARS_EXPERIENCE:
   - Found non-numeric values like 'seven', 'five', '3+', '2 years', '—'
   - Recommendation: COERCE (convert to numeric)
   - Rationale: Most values can be cleaned and converted to meaningful numbers
   - Use custom cleaning function to handle text numbers and extract numeric values

2. HOURLY_RATE:
   - Found currency symbols ($, PKR), suffixes (usd), and malformed values
   - Recommendation: COERCE (extract numeric values)
   - Rationale: Currency symbols can be stripped to get numeric values
   - Use regex to extract numbers and handle different currency formats

3. RATING:
   - Found missing values and some out-of-range values (>5)
   - Recommendation: COERCE for missing values, REVIEW out-of-range
   - Rationale: Missing ratings could be imputed with median/mean
   - Out-of-range values need manual review

4. SUBJECTS:
   - Found inconsistent casing, extra spaces, empty values
   - Recommendation: COERCE (normalize formatting)
   - Ra